# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np

alpha = 0.05

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")
df

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False
...,...,...,...,...,...,...,...,...,...,...,...
795,Diancie,Rock,Fairy,50,100,150,100,150,50,6,True
796,Mega Diancie,Rock,Fairy,50,160,110,160,110,110,6,True
797,Hoopa Confined,Psychic,Ghost,80,110,60,150,130,70,6,True
798,Hoopa Unbound,Psychic,Dark,80,160,60,170,130,80,6,True


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

**Hypotheses** (μ = mean HP, using the primary type `Type 1`)

- H0: μ_Dragon ≤ μ_Grass
- H1: μ_Dragon > μ_Grass

**Test:** two independent samples → **two-sample t-test**, one-tailed (`alternative='greater'`).
We first check the variances with Levene's test; if they differ we use Welch's version (`equal_var=False`).

In [3]:
dragon_hp = df.loc[df["Type 1"] == "Dragon", "HP"]
grass_hp  = df.loc[df["Type 1"] == "Grass",  "HP"]

print(f"Dragon: n={len(dragon_hp)}, mean HP={dragon_hp.mean():.2f}, std={dragon_hp.std():.2f}")
print(f"Grass : n={len(grass_hp)}, mean HP={grass_hp.mean():.2f}, std={grass_hp.std():.2f}")

# Check equal variances
lev_stat, lev_p = st.levene(dragon_hp, grass_hp)
equal_var = lev_p > alpha
print(f"\nLevene p-value = {lev_p:.4f} -> equal variances assumed: {equal_var}")

# One-tailed two-sample t-test
t_stat, p_value = st.ttest_ind(dragon_hp, grass_hp, equal_var=equal_var, alternative="greater")
print(f"t = {t_stat:.4f}, p-value (one-tailed) = {p_value:.6f}")

if p_value < alpha:
    print("\nReject H0: Dragon Pokemons have significantly higher average HP than Grass Pokemons.")
else:
    print("\nFail to reject H0: no evidence that Dragon Pokemons have higher average HP than Grass Pokemons.")

Dragon: n=32, mean HP=83.31, std=23.80
Grass : n=70, mean HP=67.27, std=19.52

Levene p-value = 0.1748 -> equal variances assumed: True
t = 3.5904, p-value (one-tailed) = 0.000257

Reject H0: Dragon Pokemons have significantly higher average HP than Grass Pokemons.


**Comment:** Dragon (n=32) has a mean HP of **83.31** vs **67.27** for Grass (n=70). Levene's p = 0.175, so equal variances are assumed.
The t-test gives **t = 3.59, one-tailed p ≈ 0.00026**, well below 0.05.
We **reject H0**: at the 5% significance level, the data supports the claim that Dragon Pokemons have, on average, more HP than Grass Pokemons.

- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


**Hypotheses** (for each stat)

- H0: μ_Legendary = μ_Non-Legendary
- H1: μ_Legendary ≠ μ_Non-Legendary

**Test:** two independent groups → **two-sample t-test**, two-tailed. Group sizes are very unbalanced (few Legendaries), so we use **Welch's t-test** (`equal_var=False`), which doesn't assume equal variances.

Because we run 6 tests, we also show a **Bonferroni-corrected** threshold (0.05 / 6) to control the overall false-positive rate.

In [4]:
stats_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

legendary     = df[df["Legendary"] == True]
non_legendary = df[df["Legendary"] == False]
print(f"Legendary: {len(legendary)} | Non-Legendary: {len(non_legendary)}\n")

bonferroni_alpha = alpha / len(stats_cols)
results = []
for col in stats_cols:
    t, p = st.ttest_ind(legendary[col], non_legendary[col], equal_var=False)
    results.append({
        "stat": col,
        "mean_legendary": legendary[col].mean(),
        "mean_non_legendary": non_legendary[col].mean(),
        "t_stat": t,
        "p_value": p,
        "reject_H0 (α=0.05)": p < alpha,
        f"reject_H0 (Bonferroni α={bonferroni_alpha:.4f})": p < bonferroni_alpha,
    })

results_df = pd.DataFrame(results).set_index("stat").round(4)
results_df

Legendary: 65 | Non-Legendary: 735



,mean_legendary,mean_non_legendary,t_stat,p_value,reject_H0 (α=0.05),reject_H0 (Bonferroni α=0.0083)
stat,,,,,,
HP,92.7385,67.1823,8.9814,0.0,True,True
Attack,116.6769,75.6694,10.4381,0.0,True,True
Defense,99.6615,71.5592,7.6371,0.0,True,True
Sp. Atk,122.1846,68.4544,13.4174,0.0,True,True
Sp. Def,105.9385,68.8925,10.0157,0.0,True,True
Speed,100.1846,65.4558,11.4750,0.0,True,True


In [5]:
# Mean difference per stat (positive = Legendary higher)
diff = (legendary[stats_cols].mean() - non_legendary[stats_cols].mean()).round(2)
diff.to_frame("Legendary - Non-Legendary (mean)")

,Legendary - Non-Legendary (mean)
HP,25.56
Attack,41.01
Defense,28.10
Sp. Atk,53.73
Sp. Def,37.05
Speed,34.73


**Comment:** With 65 Legendary vs 735 Non-Legendary Pokemons:

| Stat | Legendary mean | Non-Legendary mean | t | p-value |
|---|---|---|---|---|
| HP | 92.74 | 67.18 | 8.98 | 1.0e-13 |
| Attack | 116.68 | 75.67 | 10.44 | 2.5e-16 |
| Defense | 99.66 | 71.56 | 7.64 | 4.8e-11 |
| Sp. Atk | 122.18 | 68.45 | 13.42 | 1.6e-21 |
| Sp. Def | 105.94 | 68.89 | 10.02 | 2.3e-15 |
| Speed | 100.18 | 65.46 | 11.48 | 1.0e-18 |

For every one of the six stats the p-value is far below 0.05, and still below the stricter Bonferroni threshold (0.0083).
We **reject H0** for all stats: Legendary Pokemons have significantly different stats from Non-Legendary ones, and the mean differences show they are consistently **higher** across the board.

**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [6]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [7]:
school   = (-118, 34)
hospital = (-122, 37)

def euclidean_distance(lon, lat, point):
    """Euclidean distance between (lon, lat) and a (lon, lat) point. Works on scalars or Series."""
    return np.sqrt((lon - point[0])**2 + (lat - point[1])**2)

df["dist_school"]   = euclidean_distance(df["longitude"], df["latitude"], school)
df["dist_hospital"] = euclidean_distance(df["longitude"], df["latitude"], hospital)

# Close if within 0.50 of the school OR the hospital
df["close"] = (df["dist_school"] < 0.5) | (df["dist_hospital"] < 0.5)

close_prices = df.loc[df["close"],  "median_house_value"]
far_prices   = df.loc[~df["close"], "median_house_value"]

print(f"Close: n={len(close_prices)}, mean value = {close_prices.mean():,.0f}")
print(f"Far  : n={len(far_prices)}, mean value = {far_prices.mean():,.0f}")

Close: n=6829, mean value = 246,952
Far  : n=10171, mean value = 180,678


**Hypotheses** (μ = mean `median_house_value`)

- H0: μ_close ≤ μ_far
- H1: μ_close > μ_far

**Test:** two independent groups → **two-sample t-test**, one-tailed (`alternative='greater'`), Welch's version (groups have different sizes and variances).

In [8]:
lev_stat, lev_p = st.levene(close_prices, far_prices)
print(f"Levene p-value = {lev_p:.4g} -> equal variances assumed: {lev_p > alpha}")

t_stat, p_value = st.ttest_ind(close_prices, far_prices, equal_var=False, alternative="greater")
print(f"t = {t_stat:.4f}, p-value (one-tailed) = {p_value:.4g}")

if p_value < alpha:
    print("\nReject H0: houses close to a school or hospital are significantly more expensive.")
else:
    print("\nFail to reject H0: no evidence that houses close to a school or hospital are more expensive.")

Levene p-value = 0.0208 -> equal variances assumed: False
t = 37.9923, p-value (one-tailed) = 1.503e-301

Reject H0: houses close to a school or hospital are significantly more expensive.


**Comment:** 6,829 neighborhoods are close to the school or hospital and 10,171 are far. Close houses have a mean value of **\$246,952** vs **\$180,678** for far ones (about \$66k, or 37%, more).
Levene's p = 0.021, so variances differ and Welch's test is the right choice. It gives **t = 37.99, one-tailed p ≈ 1.5e-301**, essentially zero.
We **reject H0**: at the 5% significance level, houses close to a school or hospital are more expensive.

*Considerations:* the test shows an association, not causation — the two reference points sit in the Los Angeles and San Francisco Bay areas, so "closeness" also captures being in large, expensive metro areas.